# 2 — Risk differentiation : modélisation (segment SB)

Ce notebook construit le modèle de scoring du segment SB à partir des 37 variables retenues au notebook 1. Il commence par le modèle concurrent (XGBoost), qui fixe un niveau de performance de référence, puis construit la régression logistique, modèle retenu pour la grille de score.

## 2.1 Modèle concurrent : XGBoost (5.2)

XGBoost capte les non-linéarités et les interactions sans contrainte : son Gini donne une idée du plafond atteignable avec ces données. Il est estimé deux fois :
- sur les **valeurs brutes** des variables : plafond de performance ;
- sur la **WOE des classes finales** : même information que la régression logistique, ce qui isole l'effet du découpage.

L'arrêt anticipé se fait sur 20 % des clients de l'apprentissage (tirage par client) ; le test n'est utilisé que pour la mesure finale. XGBoost reste un modèle de comparaison : il n'est pas retenu comme modèle final en scoring réglementaire.


In [2]:
# Cellule 1 — paramètres et chargement
from pathlib import Path
import polars as pl
from utils import binning, models
from utils.segmentation import split_train_test

pl.Config.set_tbl_rows(-1); pl.Config.set_tbl_cols(-1); pl.Config.set_tbl_width_chars(200)
pl.Config.set_tbl_hide_column_data_types(True)

ID, TARGET, DATE = "gdt", "ddefaut_ndb", "datdelhis"
CODE = "SB"
base = pl.read_parquet(f"data/processed/base_modele_{CODE}.parquet")
regles = binning.charger_regles(f"outputs/scorecard/regles_{CODE}.json")
selection = pl.read_excel(f"outputs/binning_{CODE}.xlsx", sheet_name=f"{CODE}_Selection")
VARS_MODELE = selection.filter(pl.col("statut") == "retenue")["variable"].to_list()

train, test = base.filter(pl.col("echantillon") == "train"), base.filter(pl.col("echantillon") == "test")
print(len(VARS_MODELE), "variables |", train.height, "lignes train |", test.height, "lignes test")


37 variables | 88629 lignes train | 37996 lignes test


In [3]:
# Cellule 2 — échantillon de validation pour l'arrêt anticipé (20 % des clients du train)
train_v = split_train_test(train.drop("echantillon"), ID, TARGET, [DATE], part_test=0.2, seed=1)
fit = train_v.filter(pl.col("echantillon") == "train")
valid = train_v.filter(pl.col("echantillon") == "test")
print(fit.height, "lignes d'estimation |", valid.height, "lignes de validation")


70909 lignes d'estimation | 17720 lignes de validation


In [4]:
# Cellule 3 — XGBoost sur valeurs brutes et sur WOE, mesuré sur le train complet et le test
resultats, modeles = [], {}
for enc in ("brut", "woe"):
    X_fit, X_valid, X_train, X_test = (models.matrice_xgb(d, VARS_MODELE, regles, TARGET, enc, ref=train)
                                        for d in (fit, valid, train, test))
    m = models.entrainer_xgb(X_fit, fit[TARGET], X_valid, valid[TARGET])
    modeles[enc] = (m, X_test)
    resultats += [models.metriques(train[TARGET], m.predict_proba(X_train)[:, 1], f"XGBoost {enc} — train"),
                  models.metriques(test[TARGET], m.predict_proba(X_test)[:, 1], f"XGBoost {enc} — test")]
    print(enc, ": arrêt à", m.best_iteration + 1, "arbres")
pl.DataFrame(resultats).select("echantillon", "auc", "gini", "ks")


brut : arrêt à 328 arbres
woe : arrêt à 198 arbres


shape: (4, 4)
┌──────────────────────┬──────────┬──────────┬──────────┐
│ echantillon          ┆ auc      ┆ gini     ┆ ks       │
╞══════════════════════╪══════════╪══════════╪══════════╡
│ XGBoost brut — train ┆ 0.947692 ┆ 0.895383 ┆ 0.74946  │
│ XGBoost brut — test  ┆ 0.930085 ┆ 0.86017  ┆ 0.722353 │
│ XGBoost woe — train  ┆ 0.9342   ┆ 0.868401 ┆ 0.725455 │
│ XGBoost woe — test   ┆ 0.922619 ┆ 0.845239 ┆ 0.706514 │
└──────────────────────┴──────────┴──────────┴──────────┘

In [5]:
# Cellule 4 — importance SHAP (XGBoost brut, sur le test) confrontée à l'IV
m, X_test = modeles["brut"]
shap = models.importance_shap(m, X_test, VARS_MODELE)
models.confronter_iv_shap(selection, shap)


shape: (37, 6)
┌───────────────────────────────┬──────────┬───────────┬─────────┬───────────┬────────────┐
│ variable                      ┆ iv_test  ┆ part_shap ┆ rang_iv ┆ rang_shap ┆ ecart_rang │
╞═══════════════════════════════╪══════════╪═══════════╪═════════╪═══════════╪════════════╡
│ MINSOLDE_PAR                  ┆ 1.495637 ┆ 0.111209  ┆ 5       ┆ 1         ┆ 4          │
│ CRTAE_AG_SLDEPA               ┆ 1.087963 ┆ 0.102273  ┆ 9       ┆ 2         ┆ 7          │
│ TX_UTIL_DAV                   ┆ 1.72771  ┆ 0.08043   ┆ 3       ┆ 3         ┆ 0          │
│ bilan                         ┆ 1.372296 ┆ 0.07665   ┆ 7       ┆ 4         ┆ 3          │
│ solde_crediteur               ┆ 1.73223  ┆ 0.067321  ┆ 2       ┆ 5         ┆ -3         │
│ ARR_GDT                       ┆ 1.689217 ┆ 0.066277  ┆ 4       ┆ 6         ┆ -2         │
│ horsbilan                     ┆ 1.754623 ┆ 0.062767  ┆ 1       ┆ 7         ┆ -6         │
│ anciennete_jour_sain_M0       ┆ 0.61431  ┆ 0.045755  ┆ 17      